# US100 Index Modelling Study

**Purpose.** Model a hypothetical 100-stock, free-float market-cap-weighted index and estimate the holdings and rebalance flows of passive funds tracking 4.3% of the index's float capitalisation.

**Run instructions.** Keep `US100_constituents (1) (1).csv` in the same folder as this notebook and select **Kernel > Restart & Run All**. The core model uses Python and pandas. ADV observations are cached below so the submitted results remain reproducible without live internet access.


## Methodology and assumptions

- Float shares = shares outstanding x free float.
- Float market cap = price x float shares.
- Index weight = security float market cap / total index float market cap.
- Tracking AUM = tracking factor x total index float market cap.
- Passive shares held = tracking factor x float shares.
- For the AAPL deletion, total tracking AUM is held constant at the event instant and redistributed across the remaining 99 constituents.
- Flow sign convention = new holding minus old holding. Negative values are sells.

> The brief's wording that weight equals float market cap divided by tracking AUM is dimensionally inconsistent. The model uses the standard index-weight definition above; tracking AUM is used only to convert weight changes into dollar flows.

**Primary methodological source:** Intropic Tracking Guide.


## Build the index


In [1]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("US100_constituents (1) (1).csv")
TRACKING_FACTOR = 0.043

df = pd.read_csv(DATA_FILE, index_col=0)
required = {"ticker", "price_usd", "shares_outstanding", "free_float"}
assert required.issubset(df.columns)
assert len(df) == 100

df["float_shares"] = df["shares_outstanding"] * df["free_float"]
df["float_market_cap"] = df["price_usd"] * df["float_shares"]
TOTAL_FLOAT_CAP = df["float_market_cap"].sum()
df["weight"] = df["float_market_cap"] / TOTAL_FLOAT_CAP
TRACKING_AUM = TOTAL_FLOAT_CAP * TRACKING_FACTOR

print(f"Constituents: {len(df)}")
print(f"Total float market cap: ${TOTAL_FLOAT_CAP:,.0f}")
print(f"Weight check: {df['weight'].sum():.6%}")


Constituents: 100
Total float market cap: $21,864,916,552,920
Weight check: 100.000000%


## Q1. Total AUM tracking US100


In [ ]:
tracking_aum = TOTAL_FLOAT_CAP * TRACKING_FACTOR
print(f"Tracking factor: {TRACKING_FACTOR:.1%}")
print(f"AUM tracking US100: ${tracking_aum:,.2f} (${tracking_aum/1e9:,.2f}bn)")


Tracking factor: 4.3%
AUM tracking US100: $940,191,411,775.56 ($940.19bn)


## Q2. Current RTX weight


In [ ]:
rtx = df.loc[df["ticker"].eq("RTX")].iloc[0]
print(f"RTX float market cap: ${rtx['float_market_cap']:,.2f}")
print(f"RTX index weight: {rtx['weight']:.4%}")


RTX float market cap: $137,227,355,280.00
RTX index weight: 0.6276%


## Q3. GOOG held by passive funds


In [ ]:
goog = df.loc[df["ticker"].eq("GOOG")].iloc[0]
goog_passive_shares = goog["float_shares"] * TRACKING_FACTOR
goog_passive_usd = goog_passive_shares * goog["price_usd"]

print(f"GOOG passive holding (shares): {goog_passive_shares:,.0f}")
print(f"GOOG passive holding (USD): ${goog_passive_usd:,.2f}")


GOOG passive holding (shares): 11,889,500
GOOG passive holding (USD): $26,795,247,255.00


## Q4. ORCL weight at 80.3% free float


In [ ]:
orcl = df.loc[df["ticker"].eq("ORCL")].iloc[0]
new_free_float = 0.803
orcl_new_float_cap = orcl["price_usd"] * orcl["shares_outstanding"] * new_free_float
new_total_float_cap = TOTAL_FLOAT_CAP - orcl["float_market_cap"] + orcl_new_float_cap
orcl_new_weight = orcl_new_float_cap / new_total_float_cap

print(f"ORCL old weight: {orcl['weight']:.4%}")
print(f"ORCL new float market cap: ${orcl_new_float_cap:,.2f}")
print(f"ORCL new index weight: {orcl_new_weight:.4%}")


ORCL old weight: 0.4980%
ORCL new float market cap: $144,999,396,300.00
ORCL new index weight: 0.6621%


## Q5. Top 10 absolute flows after deleting AAPL

AAPL must be included because its move from its current weight to zero is the largest absolute flow. The remaining AAPL sale is redistributed pro rata across the other 99 stocks.


In [ ]:
aapl = df.loc[df["ticker"].eq("AAPL")].iloc[0]
remaining = df.loc[~df["ticker"].eq("AAPL")].copy()
remaining["new_weight"] = remaining["float_market_cap"] / remaining["float_market_cap"].sum()
remaining["weight_change"] = remaining["new_weight"] - remaining["weight"]
remaining["flow_usd"] = remaining["weight_change"] * TRACKING_AUM
remaining["flow_shares"] = remaining["flow_usd"] / remaining["price_usd"]

aapl_flow = pd.DataFrame([{
    "ticker": "AAPL",
    "price_usd": aapl["price_usd"],
    "weight": aapl["weight"],
    "new_weight": 0.0,
    "weight_change": -aapl["weight"],
    "flow_usd": -aapl["weight"] * TRACKING_AUM,
    "flow_shares": (-aapl["weight"] * TRACKING_AUM) / aapl["price_usd"],
}])

flows = pd.concat([
    aapl_flow,
    remaining[["ticker", "price_usd", "weight", "new_weight",
               "weight_change", "flow_usd", "flow_shares"]]
], ignore_index=True)

top10 = flows.loc[flows["flow_usd"].abs().nlargest(10).index].copy()
assert abs(flows["flow_usd"].sum()) < 0.01

top10[[
    "ticker", "weight", "new_weight", "weight_change",
    "flow_usd", "flow_shares",
]].rename(columns={
    "ticker": "Ticker",
    "weight": "Old weight",
    "new_weight": "New weight",
    "weight_change": "Weight change",
    "flow_usd": "Flow (USD)",
    "flow_shares": "Flow (shares)",
})


,Ticker,Old weight,New weight,Weight change,Flow (USD),Flow (shares)
0,AAPL,0.097524,0.000000,-0.097524,-9.169106e+10,-6.631305e+08
1,MSFT,0.089172,0.098808,0.009636,9.059801e+09,3.499885e+07
4,AMZN,0.044606,0.049426,0.004820,4.531914e+09,4.030518e+07
2,GOOGL,0.030820,0.034150,0.003330,3.131282e+09,1.394880e+06
3,GOOG,0.028500,0.031580,0.003080,2.895561e+09,1.284809e+06
5,TSLA,0.026813,0.029711,0.002897,2.724197e+09,3.862958e+06
6,BRK/B,0.022403,0.024824,0.002421,2.276157e+09,8.508362e+06
7,JNJ,0.021613,0.023949,0.002336,2.195893e+09,1.222249e+07
8,UNH,0.021493,0.023815,0.002323,2.183633e+09,4.368927e+06
10,NVDA,0.017738,0.019655,0.001917,1.802182e+09,1.110744e+07


<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 880 330" role="img" aria-label="Top 10 absolute flows after AAPL deletion">
<rect width="100%" height="100%" fill="white"/>
<text x="95" y="25" font-family="Arial" font-size="16" font-weight="700" fill="#101828">Top 10 absolute flows after AAPL deletion</text>
<text x="85" y="63" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">AAPL</text>
<rect x="95" y="48" width="635.0" height="20" rx="2" fill="#B42318"/>
<text x="738.0" y="63" font-family="Arial" font-size="11" fill="#344054">-91.69bn</text>
<text x="85" y="90" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">MSFT</text>
<rect x="95" y="75" width="62.7" height="20" rx="2" fill="#175CD3"/>
<text x="165.7" y="90" font-family="Arial" font-size="11" fill="#344054">+9.06bn</text>
<text x="85" y="117" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">AMZN</text>
<rect x="95" y="102" width="31.4" height="20" rx="2" fill="#175CD3"/>
<text x="134.4" y="117" font-family="Arial" font-size="11" fill="#344054">+4.53bn</text>
<text x="85" y="144" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">GOOGL</text>
<rect x="95" y="129" width="21.7" height="20" rx="2" fill="#175CD3"/>
<text x="124.7" y="144" font-family="Arial" font-size="11" fill="#344054">+3.13bn</text>
<text x="85" y="171" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">GOOG</text>
<rect x="95" y="156" width="20.1" height="20" rx="2" fill="#175CD3"/>
<text x="123.1" y="171" font-family="Arial" font-size="11" fill="#344054">+2.90bn</text>
<text x="85" y="198" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">TSLA</text>
<rect x="95" y="183" width="18.9" height="20" rx="2" fill="#175CD3"/>
<text x="121.9" y="198" font-family="Arial" font-size="11" fill="#344054">+2.72bn</text>
<text x="85" y="225" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">BRK/B</text>
<rect x="95" y="210" width="15.8" height="20" rx="2" fill="#175CD3"/>
<text x="118.8" y="225" font-family="Arial" font-size="11" fill="#344054">+2.28bn</text>
<text x="85" y="252" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">JNJ</text>
<rect x="95" y="237" width="15.2" height="20" rx="2" fill="#175CD3"/>
<text x="118.2" y="252" font-family="Arial" font-size="11" fill="#344054">+2.20bn</text>
<text x="85" y="279" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">UNH</text>
<rect x="95" y="264" width="15.1" height="20" rx="2" fill="#175CD3"/>
<text x="118.1" y="279" font-family="Arial" font-size="11" fill="#344054">+2.18bn</text>
<text x="85" y="306" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">NVDA</text>
<rect x="95" y="291" width="12.5" height="20" rx="2" fill="#175CD3"/>
<text x="115.5" y="306" font-family="Arial" font-size="11" fill="#344054">+1.80bn</text>
</svg>


## Q6a. Flow pressure in ADV

**Source:** Yahoo Finance chart endpoint, daily share volume. ADV is the arithmetic mean of the latest 30 trading sessions available from **2026-05-07 to 2026-06-18**, retrieved 21 June 2026.

ADV is time-sensitive. For an actual rebalance forecast, the window should end immediately before the announced trade date. The brief does not provide a trade date, so a clearly timestamped current window is used.


In [ ]:
# Window: 2026-05-07 to 2026-06-18; retrieved 2026-06-21.
# Yahoo ticker BRK-B is mapped back to the case-study ticker BRK/B.
ADV_30D = {'AAPL': 49931193.333333, 'AMZN': 41622950.0, 'BRK/B': 5283960.0, 'GOOG': 21491810.0, 'GOOGL': 30767840.0, 'JNJ': 8078033.333333, 'MSFT': 37415286.666667, 'NVDA': 170728256.666667, 'TSLA': 50621556.666667, 'UNH': 7104010.0}

def fetch_yahoo_adv(tickers, end_date=None, sessions=30):
    """Optional refresh using Yahoo's public chart endpoint.

    Set end_date to the day before an actual rebalance trade date. If omitted,
    the latest available observations are used. This function requires internet.
    """
    import datetime as dt
    import json
    import urllib.parse
    import urllib.request

    if end_date is None:
        end = dt.datetime.now(dt.timezone.utc)
    else:
        end = dt.datetime.combine(
            dt.date.fromisoformat(end_date),
            dt.time.max,
            tzinfo=dt.timezone.utc,
        )
    start = end - dt.timedelta(days=120)
    result = {}
    for ticker in tickers:
        yahoo_ticker = ticker.replace("/", "-")
        query = urllib.parse.urlencode({
            "period1": int(start.timestamp()),
            "period2": int(end.timestamp()),
            "interval": "1d",
        })
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{yahoo_ticker}?{query}"
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as response:
            chart = json.load(response)["chart"]["result"][0]
        volumes = [v for v in chart["indicators"]["quote"][0]["volume"] if v is not None]
        if len(volumes) < sessions:
            raise ValueError(f"{ticker}: only {len(volumes)} valid sessions available")
        result[ticker] = sum(volumes[-sessions:]) / sessions
    return result

top10["adv_30d_shares"] = top10["ticker"].map(ADV_30D)
top10["flow_adv"] = top10["flow_shares"].abs() / top10["adv_30d_shares"]

adv_table = top10[[
    "ticker", "flow_shares", "adv_30d_shares", "flow_adv",
]].copy()
adv_table["flow_shares"] = adv_table["flow_shares"].abs()
adv_table = adv_table.rename(columns={
    "ticker": "Ticker",
    "flow_shares": "Absolute flow shares",
    "adv_30d_shares": "30-day ADV",
    "flow_adv": "Flow / ADV",
})
adv_table


,Ticker,Absolute flow shares,30-day ADV,Flow / ADV
0,AAPL,6.631305e+08,4.993119e+07,13.280887
1,MSFT,3.499885e+07,3.741529e+07,0.935416
4,AMZN,4.030518e+07,4.162295e+07,0.968340
2,GOOGL,1.394880e+06,3.076784e+07,0.045336
3,GOOG,1.284809e+06,2.149181e+07,0.059781
5,TSLA,3.862958e+06,5.062156e+07,0.076311
6,BRK/B,8.508362e+06,5.283960e+06,1.610225
7,JNJ,1.222249e+07,8.078033e+06,1.513053
8,UNH,4.368927e+06,7.104010e+06,0.614994
10,NVDA,1.110744e+07,1.707283e+08,0.065059


<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 880 330" role="img" aria-label="Flow pressure as a multiple of 30-day ADV">
<rect width="100%" height="100%" fill="white"/>
<text x="95" y="25" font-family="Arial" font-size="16" font-weight="700" fill="#101828">Flow pressure as a multiple of 30-day ADV</text>
<text x="85" y="63" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">AAPL</text>
<rect x="95" y="48" width="635.0" height="20" rx="2" fill="#7F56D9"/>
<text x="738.0" y="63" font-family="Arial" font-size="11" fill="#344054">13.28x</text>
<text x="85" y="90" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">MSFT</text>
<rect x="95" y="75" width="44.7" height="20" rx="2" fill="#7F56D9"/>
<text x="147.7" y="90" font-family="Arial" font-size="11" fill="#344054">0.94x</text>
<text x="85" y="117" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">AMZN</text>
<rect x="95" y="102" width="46.3" height="20" rx="2" fill="#7F56D9"/>
<text x="149.3" y="117" font-family="Arial" font-size="11" fill="#344054">0.97x</text>
<text x="85" y="144" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">GOOGL</text>
<rect x="95" y="129" width="2.2" height="20" rx="2" fill="#7F56D9"/>
<text x="105.2" y="144" font-family="Arial" font-size="11" fill="#344054">0.05x</text>
<text x="85" y="171" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">GOOG</text>
<rect x="95" y="156" width="2.9" height="20" rx="2" fill="#7F56D9"/>
<text x="105.9" y="171" font-family="Arial" font-size="11" fill="#344054">0.06x</text>
<text x="85" y="198" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">TSLA</text>
<rect x="95" y="183" width="3.6" height="20" rx="2" fill="#7F56D9"/>
<text x="106.6" y="198" font-family="Arial" font-size="11" fill="#344054">0.08x</text>
<text x="85" y="225" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">BRK/B</text>
<rect x="95" y="210" width="77.0" height="20" rx="2" fill="#7F56D9"/>
<text x="180.0" y="225" font-family="Arial" font-size="11" fill="#344054">1.61x</text>
<text x="85" y="252" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">JNJ</text>
<rect x="95" y="237" width="72.3" height="20" rx="2" fill="#7F56D9"/>
<text x="175.3" y="252" font-family="Arial" font-size="11" fill="#344054">1.51x</text>
<text x="85" y="279" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">UNH</text>
<rect x="95" y="264" width="29.4" height="20" rx="2" fill="#7F56D9"/>
<text x="132.4" y="279" font-family="Arial" font-size="11" fill="#344054">0.61x</text>
<text x="85" y="306" text-anchor="end" font-family="Arial" font-size="12" fill="#344054">NVDA</text>
<rect x="95" y="291" width="3.1" height="20" rx="2" fill="#7F56D9"/>
<text x="106.1" y="306" font-family="Arial" font-size="11" fill="#344054">0.07x</text>
</svg>


## Q6b. Effect of a 20-for-1 AMZN stock split


In [ ]:
amzn = df.loc[df["ticker"].eq("AMZN")].iloc[0]
before_shares = amzn["float_shares"] * TRACKING_FACTOR
before_value = before_shares * amzn["price_usd"]
after_shares = before_shares * 20
after_price = amzn["price_usd"] / 20
after_value = after_shares * after_price

print(f"Before: {before_shares:,.0f} shares at ${amzn['price_usd']:,.2f} = ${before_value:,.2f}")
print(f"After:  {after_shares:,.0f} shares at ${after_price:,.4f} = ${after_value:,.2f}")
print("Index weight and dollar exposure are unchanged; no rebalance trade is required.")


Before: 372,980,280 shares at $112.44 = $41,937,902,683.20
After:  7,459,605,600 shares at $5.6220 = $41,937,902,683.20
Index weight and dollar exposure are unchanged; no rebalance trade is required.


## Validation


In [9]:
# Validation checks
assert len(df) == 100
assert abs(df["weight"].sum() - 1.0) < 1e-12
assert abs(remaining["new_weight"].sum() - 1.0) < 1e-12
assert abs(flows["flow_usd"].sum()) < 0.01
assert abs(remaining["flow_usd"].sum() + aapl_flow["flow_usd"].iloc[0]) < 0.01
assert top10.iloc[0]["ticker"] == "AAPL"
assert abs(before_value - after_value) < 0.01
print("All validation checks passed.")


All validation checks passed.


## Tracking-factor sensitivity

Index weights are independent of the tracking assumption, but passive AUM and rebalance dollar flows scale linearly with it. This table shows the commercial impact if the 4.3% estimate is revised.


In [10]:
# Commercial sensitivity: weights do not change, but AUM and dollar flows scale
# linearly with the assumed tracking factor.
sensitivity_factors = [0.03, 0.043, 0.05, 0.07]
pd.DataFrame({
    "Tracking factor": [f"{factor:.1%}" for factor in sensitivity_factors],
    "Tracking AUM": [f"${TOTAL_FLOAT_CAP * factor / 1e9:,.2f}bn"
                     for factor in sensitivity_factors],
    "AAPL deletion sell": [f"${-aapl['weight'] * TOTAL_FLOAT_CAP * factor / 1e9:,.2f}bn"
                           for factor in sensitivity_factors],
})


,Tracking factor,Tracking AUM,AAPL deletion sell
0,3.0%,$655.95bn,$-63.97bn
1,4.3%,$940.19bn,$-91.69bn
2,5.0%,"$1,093.25bn",$-106.62bn
3,7.0%,"$1,530.54bn",$-149.26bn


## Final answers

| Question | Answer |
|---|---:|
| 1. Tracking AUM | **$940,191,411,775.56 ($940.19bn)** |
| 2. RTX weight | **0.6276%** |
| 3a. GOOG passive value | **$26,795,247,255.00** |
| 3b. GOOG passive shares | **11,889,500** |
| 4. ORCL weight at 80.3% float | **0.6621%** |
| 5. Largest deletion flow | **AAPL: $-91,691,057,000.40 (-663,130,520 shares)** |
| 6a. Largest ADV pressure | **AAPL: 13.28x ADV** |
| 6b. AMZN split | **Shares x20; price /20; dollar exposure and index weight unchanged** |
